In [0]:
%python
display(dbutils.fs.ls("/Volumes/aman/bookstore/data/"))

In [0]:
create or replace table customers 
AS 
select * from json.`dbfs:/Volumes/aman/bookstore/data/customers-json/`

In [0]:
select * from customers
limit 4

In [0]:
describe customers

### How To extract json sparsed file?

# 1. Accessing Json  nested column

###Method 1: we can use ':', column_name:level_1..level_n

**Profile** is column name but have first_name, last_name, gender and address object where address have its own nested name in it.

In [0]:
select profile from customers
limit 1

In [0]:
select 
  customer_id,
  profile:first_name,
  profile:address:country
from customers

### Method 2: from_json.

Here we define the STRUCT Type of our json file and use . method to access them

In [0]:
SELECT
    parsed.first_name,
    parsed.last_name,
    parsed.gender,
    parsed.address.street,
    parsed.address.city,
    parsed.address.country
FROM (
    SELECT from_json(
        profile,
        'STRUCT<
            first_name: STRING,
            last_name: STRING,
            gender: STRING,
            address: STRUCT<
                street: STRING,
                city: STRING,
                country: STRING
            >
        >'
    ) AS parsed
    FROM customers
);

### Method 3: Get_json_object

In [0]:
select customer_id,
    get_json_object(profile, '$.first_name') as first_name,
    get_json_object(profile, '$.address.country') as country
from customers

#2. Advanced Json Method

## Order Table

In [0]:
CREATE OR REPLACE table aman.delta.orders AS
SELECT *
FROM json.`dbfs:/Volumes/aman/bookstore/data/orders-json-raw/`

In [0]:
select *
from aman.delta.orders
where customer_id = 'C00001'

In [0]:
UPDATE aman.delta.orders
SET books = ARRAY(
    NAMED_STRUCT('book_id', 'B01', 'quantity', 1, 'subtotal', 49),
    NAMED_STRUCT('book_id', 'B02', 'quantity', 1, 'subtotal', 35)
)
WHERE customer_id = 'C00762';

In [0]:
select 
       order_id,
       customer_id,
       explode(books) as book
from orders
where customer_id = 'C00644'

In [0]:
select * from aman.delta.orders
where customer_id = 'C00644'

In [0]:
select  
      customer_id,
      collect_list(order_id) as order_set,
      collect_list(books.book_id) as book_set
from orders
group by customer_id

In [0]:
select  
      customer_id,
      collect_list(books.book_id) as book_set,
      array_distinct(flatten(collect_list(books.book_id))) As after_flatten
from orders
group by customer_id

#4. Explode, Split, Array, Array Contains()

In [0]:
%python
data = [
    (1, "Amanuel", ["PySpark", "Databricks", "SQL", "Python"]),
    (2, "Rahel", ["SAP", "PowerBI", "JavaScript"]),
    (3, "Tinsae", ["Management", "Design", "Construction"])
]

schema = ["id", "name", "skills"]

df = spark.createDataFrame(data, schema)

df.show()

##4.1 Explode Function

In [0]:
%python
df.printSchema()

In [0]:
%python 
from pyspark.sql.functions import *

In [0]:
%python
help(display)

In [0]:
%python
display(df,truncate=False)
df1= df.withColumn('skillExplode',explode(col('skills'))).select('id','name','skillExplode')
df1.show()

# Collect List

In [0]:
%python
display(df)
display(df1)
df2 = df1.groupBy('id','name').agg(collect_list('skillExplode').alias('skillSet'))
display(df2)

In [0]:
%python
df1.printSchema()

## 4.2. Split

The split column used to collect string element as an array with using delimiter.

In [0]:
%python
data = [
    (1, "Amanuel", "PySpark, Databricks, SQL, Python"),
    (2, "Rahel", "SAP,PowerBI, JavaScript"),
    (3, "Tinsae", "Management, Design, Construction")
]

schema = ["id", "name", "skills"]

df = spark.createDataFrame(data, schema)

df.show()

In [0]:
%python
display(df)
df1 = df.withColumn('skillSet',split(col('skills'),','))
display(df1)

## 4.3 Array ()

we can use array() to collect set of elements from columns value

In [0]:
%python

data = [
    (1, "Amanuel", "PySpark", "Databricks", "SQL", "Python"),
    (2, "Rahel", "SAP", "PowerBI", "JavaScript", "SQL"),
    (3, "Tinsae", "Management", "Design", "Construction", None)
]

schema = [
    "id",
    "name",
    "primary_skill",
    "secondary_skill",
    "third_skill",
    "fourth_skill"
]

df = spark.createDataFrame(data, schema)

df.show()

In [0]:
%python
display(df)
df3 = df.withColumn('CollectedSkillSet',array(col('primary_skill'),col('secondary_skill'),col('third_skill'),col('fourth_skill'))).select('id','name','CollectedSkillSet')
display(df3)

## 4.4 Array_Contains

In [0]:
%python
df4 = df3.withColumn('SQLSkill', coalesce(
        array_contains(col("CollectedSkillSet"), "SQL"),
        lit(False)
    ))
display(df4.select('id','name','CollectedSkillSet','SQLSkill'))

In [0]:
%python
df.write.mode("overwrite").saveAsTable("df")
